# I4.1 — Convergence study on M03

Exercise card: `I4.1_convergence_study.md` · 10 min in the session (part 1, questions 1–2); part 2
and the extension after the session.

**Goal.** Separate the error of the model (number of nodes) from the error of the solver
(tolerance), and choose both for a purpose.

The case is `EnergyTraining.Examples.S04_StorageCharging`: M03, 300 L at 30 °C, charged from the
top with 0.1 kg/s at 60 °C for 3 h. $N$ sets the size of the arrays, so each $N$ is a new model:
`simulate_variant` writes it, compiles it and returns the result and the solver statistics.
Cells tagged `omc` run OpenModelica from Python with `energy_training.omc`: a local `omc` if OpenModelica is installed, otherwise the course Docker image. Each simulation takes a few seconds (the model is translated to C and compiled first).

Complete the cells marked **TODO** and write your answers in the cells marked *Your answer*.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from energy_training.omc.runner import simulate_variant
from energy_training.omc.studies import M03_EXAMPLE, reference_outlet
from energy_training.plotting.style import apply_style, degc, hours
from energy_training.simulation import storage as st
from energy_training.utilities import repo_root

apply_style()
OUT = repo_root() / "output/s04_i1"  # git-ignored
STOP, INTERVAL = 3 * 3600.0, 60.0  # 3 h, results every minute
NODE_COUNTS = (1, 2, 5, 10, 20, 50)
TOLERANCES = (1e-2, 1e-4, 1e-6, 1e-8, 1e-10)


def stored_energy(result, n):
    """Energy stored in the tank between the first and the last sample [J]."""
    c_node = st.StorageParams(N=n).C_node
    cols = [f"storage.tank.T[{k}]" for k in range(1, n + 1)]
    return float(c_node * (result[cols].iloc[-1] - result[cols].iloc[0]).sum())


def first_time_above(series, T_C):
    """First time [s] at which a temperature series [K] exceeds T_C degC (None if never)."""
    above = series[degc(series) > T_C]
    return float(above.index[0]) if len(above) else None

## Part 1 — number of nodes

Complete `node_count_study`: one row per $N$ (DASSL, tolerance 1e-6) with the outlet temperature
after 1 h (°C), the energy stored after 3 h (kWh), the time at which water above 55 °C first
reaches the outlet (min) and the number of solver steps; also return the outlet series for the plot.
`run.result` is a DataFrame indexed by time [s]; `run.stats` a dict of solver statistics.

In [ ]:
def node_count_study(node_counts=NODE_COUNTS):
    """Part 1: (table with one row per N, {N: outlet temperature series [K]})."""
    rows, outlets = [], {}
    for n in node_counts:
        run = simulate_variant(M03_EXAMPLE, {"N": n}, out_dir=OUT, stop_time=STOP,
                               interval=INTERVAL)  # fmt: skip
        # TODO: append one row per N to rows and store run.result["storage.T_out"] in outlets
        raise NotImplementedError("TODO: complete node_count_study()")
    return pd.DataFrame(rows).set_index("N"), outlets

In [ ]:
nodes, outlets = node_count_study()
print(nodes.round(3).to_string())

fig, ax = plt.subplots()
for n, t_out in outlets.items():
    ax.plot(hours(t_out.index), degc(t_out), label=f"N = {n}")
ax.axhline(55, ls=":", color="k", lw=1)
ax.set(xlabel="time [h]", ylabel="outlet temperature [°C]", title="M03 charging, 0.1 kg/s")
ax.legend(fontsize=8);

## Question 2 — which quantity converges?

Which quantity converges quickly with $N$, and which does not? Choose $N$ for two purposes and
justify each choice in two sentences: (a) the energy a heat pump puts into the tank in 3 h;
(b) the time at which water above 55 °C first reaches the outlet.

*Your answer:*

## Part 2 — tolerance (after the session)

Complete `tolerance_study` for your $N$ of (a): DASSL tolerances 1e-2 … 1e-10; the largest error
of $T_{out}$ against `reference_outlet(n, t)` (the Python M03 with a tight tolerance, at the
times `t`) and the cost (`run.stats["f_calls"]`, right-hand-side evaluations). Plot both against
the tolerance. Is the error equal to the tolerance?

In [ ]:
def tolerance_study(n, tolerances=TOLERANCES, method=None):
    """Part 2: one row per tolerance with the largest error of T_out [K] and the cost."""
    t = np.arange(0.0, STOP + 1.0, INTERVAL)  # comparison times [s]
    raise NotImplementedError(
        "TODO: simulate with simulate_variant(..., tolerance=tol) and compare storage.T_out "
        "with reference_outlet(n, t)"
    )

In [ ]:
# TODO: tols = tolerance_study(n=...), print it and plot error and f_calls against the tolerance

*Your answer (questions 3 and 4):*

## Extension

Repeat part 2 with `method="cvode"` (tolerance 1e-6) and with explicit Euler (`method="euler"`,
step = output interval) for 60 s, 30 s and 10 s. How small must the Euler step be to match the
error of DASSL at tolerance 1e-6, and how many right-hand-side evaluations does that cost?

In [ ]:
# TODO (optional)

*Your answer:*